# Ingestão da Camada Silver

Notebook para leitura dos dados da camada **Bronze** e carga na camada **Silver** do Data Lakehouse.
Na Silver os dados são limpos, padronizados e com tipos de dados corretos, prontos para análise.
A camada Gold será implementada em notebook subsequente.

## 📚 Sobre este Notebook

Este notebook implementa a ingestão da **camada Silver** a partir dos dados já carregados na **camada Bronze**.

### Arquitetura Medalhão

* **📂 Staging**: Dados brutos copiados do Google Drive (notebook MVP02-Download)
* **🥉 Bronze**: Dados brutos com metadados de rastreabilidade (notebook MVP03-Bronze)
* **🥈 Silver**: Dados limpos, padronizados e com tipos corretos (este notebook)
* **🥇 Gold**: Dados agregados e prontos para consumo  (notebook MVP05-Gold)

### 🎯 Objetivo
Ler as tabelas da camada Bronze e aplicnar as seguintes transformações:

* **Normalização de nomes**: colunas em lowercase com separador `_`
* **Conversão de tipos**: strings de data → `DATE`/`TIMESTAMP`, strings numéricas → `DOUBLE`/`INT`
* **Tratamento de valores especiais**: `-` → `NULL`
* **Decimais**: trocar vírgula por ponto (padrão brasileiro → padrão internacional)
* **Metadados de rastreabilidade**: `data_carga_silver` e `origem_tabela_bronze`

In [0]:
%sql
-- Define o catálogo ativo como mvpDataEngineer no Unity Catalog.
USE CATALOG mvpDataEngineer;

-- Define o schema ativo como silver dentro do catálogo atual.
USE SCHEMA silver;

## 🥈 Etapa 1: Transformação Bronze → Silver

Lê cada tabela da camada **Bronze**, aplica limpeza e padronização, e grava na camada **Silver**.

### Transformações aplicadas:
* **Normalização de nomes**: colunas em lowercase com `_` (ex: `DATAREGISTRO` → `data_registro`)
* **Conversão de datas**: strings `dd/MM/yyyy` → `DATE` e `dd/MM/yyyy HH:mm:ss` → `TIMESTAMP`
* **Conversão de decimais**: vírgula → ponto (ex: `14,0` → `14.0` → `DOUBLE`)
* **Tratamento de valores ausentes**: `-` → `NULL`
* **Metadados de rastreabilidade**: `data_carga_silver` e `origem_tabela_bronze`
* **Aplicado filtro**: `STATUS = Solicitação Finalizada`
* **Campos removidos**: `OBSERVACAO` por não trazer informações significativas para analise e `SOLICDATA` pois é o momento que a leitura foi inserida no sistema. O campo `DATAREGISTRO` é o momento que a leitura foi realizada em campo e este é importante para as análises.

### Princípios da camada Silver:
* ✅ **Dados limpos** - tipos corretos, sem valores inválidos
* ✅ **Nomes padronizados** - lowercase com snake_case
* ✅ **Preserva rastreabilidade** - mantém metadados de staging e bronze
* ✅ **Formato Delta Lake** - formato otimizado do Databricks

In [0]:
# importando bibliotecas 
from pyspark.sql.functions import col, to_timestamp, to_date, regexp_replace, current_timestamp, lit, when

In [0]:
# =========================================
# TRANSFORMAÇÃO: leitura_pluviometros
# =========================================
# Transformações aplicadas:
#   - DATAREGISTRO: STRING "dd/MM/yyyy" para DATE
#   - MILIMETROS: vírgula decimal para ponto e STRING para DOUBLE
#   - Nomes de colunas: uppercase para lowercase com _
#   - Removido campos: OBSERVACAO e SOLICDATA. 
#   - Aplicado filtro para STATUS = Solicitação Finalizada

# Ler da camada Bronze
df_bronze = spark.table("mvpDataEngineer.bronze.leitura_pluviometros")
print(f" Registros lidos da bronze: {df_bronze.count()}")

# Filta apenas os registros cujo status for "Solicitação Finalizada"
df_bronze = df_bronze.filter(col("STATUS") == "Solicitação Finalizada")

# Aplicar transformações
df_silver = df_bronze.select(
    col("UNIDADE").alias("unidade"),
    col("ESTACAO").alias("estacao"),
    to_date(col("DATAREGISTRO"), "dd/MM/yyyy").alias("data_registro"),
    when(col("MILIMETROS") == "-", None).otherwise(regexp_replace(col("MILIMETROS"), ",", ".").cast("double")).alias("milimetros"),
    # Metadados de rastreabilidade (Staging + Bronze)
    col("data_ingestao"),
    col("origem"),
    col("arquivo_origem"),
    col("data_carga_bronze"),
    col("origem_tabela"),
    # Metadados de rastreabilidade (Silver)
    current_timestamp().alias("data_carga_silver"),
    lit("bronze.leitura_pluviometros").alias("origem_tabela_bronze")
)

# Salvar na camada Silver
df_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("mvpDataEngineer.silver.leitura_pluviometros")

row_count = df_silver.count()
print(f"  -Tabela criada: mvpDataEngineer.silver.leitura_pluviometros")
print(f"  -Registros salvos: {row_count}")
print(f"  -Colunas: {len(df_silver.columns)}")

In [0]:
# =========================================
# TRANSFORMAÇÃO: medicao_bacias
# =========================================
# Transformações aplicadas:
#   - SOLICDATA de STRING "dd/MM/yyyy HH:mm:ss" para TIMESTAMP
#   - DATAREGISTRO de STRING "dd/MM/yyyy" para DATE
#   - AGUADOCE_COLOCADO, AGUADOCE_RETIRADO, SALMOURA_COLOCADO e SALMOURA_RETIRADO transformado "-" em NULL, depois STRING  para INT
#   - SALMOURA_DENSIDADE: vírgula decimal para ponto e STRING para DOUBLE
#   - Nomes de colunas: uppercase para lowercase com _
#   - Removido campo de observeção
#   - Aplicado filtro para STATUS = Solicitação Finalizada

# Ler da camada Bronze
df_bronze = spark.table("mvpDataEngineer.bronze.medicao_bacias")
print(f" Registros lidos da bronze: {df_bronze.count()}")

# Filta apenas os registros cujo status for "Solicitação Finalizada"
df_bronze = df_bronze.filter(col("STATUS") == "Solicitação Finalizada")

df_silver = df_bronze.select(
    col("UNIDADE").alias("unidade"),
    col("BACIA").alias("bacia"),
    to_date(col("DATAREGISTRO"), "dd/MM/yyyy").alias("data_registro"),
    # Tratar "-" como NULL e converter para INT
    when(col("AGUADOCE_COLOCADO") == "-", None).otherwise(col("AGUADOCE_COLOCADO").cast("int")).alias("agua_doce_colocado"),
    when(col("AGUADOCE_RETIRADO") == "-", None).otherwise(col("AGUADOCE_RETIRADO").cast("int")).alias("agua_doce_retirado"),
    when(col("SALMOURA_COLOCADO") == "-", None).otherwise(col("SALMOURA_COLOCADO").cast("int")).alias("salmoura_colocado"),
    when(col("SALMOURA_RETIRADO") == "-", None).otherwise(col("SALMOURA_RETIRADO").cast("int")).alias("salmoura_retirado"),
    # Densidade: vírgula decimal para ponto e transforma em DOUBLE
    when(col("SALMOURA_DENSIDADE") == "-", None).otherwise(regexp_replace(col("SALMOURA_DENSIDADE"), ",", ".").cast("double")).alias("salmoura_densidade"),
    # Metadados de rastreabilidade (Staging + Bronze)
    col("data_ingestao"),
    col("origem"),
    col("arquivo_origem"),
    col("data_carga_bronze"),
    col("origem_tabela"),
    # Metadados de rastreabilidade (Silver)
    current_timestamp().alias("data_carga_silver"),
    lit("bronze.medicao_bacias").alias("origem_tabela_bronze")
)

# Salvar na camada Silver
df_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("mvpDataEngineer.silver.medicao_bacias")

row_count = df_silver.count()
print(f"  -Tabela criada: mvpDataEngineer.silver.leitura_pluviometros")
print(f"  -Registros salvos: {row_count}")
print(f"  -Colunas: {len(df_silver.columns)}")

In [0]:
# =========================================
# TRANSFORMAÇÃO: medicao_laje
# =========================================
# Transformações aplicadas:
#   - SOLICDATA de STRING "dd/MM/yyyy HH:mm:ss" para TIMESTAMP
#   - DATAMEDICAO: string "dd/MM/yyyy" para DATE
#   - REGUA_1 a REGUA_8: já estão como INT, apenas renomeados
#   - Nomes de colunas: uppercase para lowercase com _
#   - Removido campo de observeção
#   - Aplicado filtro para STATUS = Solicitação Finalizada

# Ler da camada Bronze
df_bronze = spark.table("mvpDataEngineer.bronze.medicao_laje")
print(f" Registros lidos da bronze: {df_bronze.count()}")

# Filta apenas os registros cujo status for "Solicitação Finalizada"
df_bronze = df_bronze.filter(col("STATUS") == "Solicitação Finalizada")

# Aplicar transformações
df_silver = df_bronze.select(
    col("UNIDADE").alias("unidade"),
    to_date(col("DATAMEDICAO"), "dd/MM/yyyy").alias("data_medicao"),
    col("CRISTALIZADOR").alias("cristalizador"),
    col("AREAHECTARES").alias("area_hectares"),
    col("REGUA_1").alias("regua_1"),
    col("REGUA_2").alias("regua_2"),
    col("REGUA_3").alias("regua_3"),
    col("REGUA_4").alias("regua_4"),
    col("REGUA_5").alias("regua_5"),
    col("REGUA_6").alias("regua_6"),
    col("REGUA_7").alias("regua_7"),
    col("REGUA_8").alias("regua_8"),
    # Metadados de rastreabilidade (Staging + Bronze)
    col("data_ingestao"),
    col("origem"),
    col("arquivo_origem"),
    col("data_carga_bronze"),
    col("origem_tabela"),
    # Metadados de rastreabilidade (Silver)
    current_timestamp().alias("data_carga_silver"),
    lit("bronze.medicao_laje").alias("origem_tabela_bronze")
)

# Salvar na camada Silver
df_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("mvpDataEngineer.silver.medicao_laje")

row_count = df_silver.count()
print(f"  -Tabela criada: mvpDataEngineer.silver.leitura_pluviometros")
print(f"  -Registros salvos: {row_count}")
print(f"  -Colunas: {len(df_silver.columns)}")

## 🔍 Etapa 2: Validação dos Dados Silver

### O que verificar?
* **Total de registros** por tabela (deve ser igual à Bronze)
* **Tipos de dados** convertidos corretamente
* **Valores NULL** onde antes havia `-`
* **Amostras** dos dados transformados

In [0]:
%sql
-- Resumo das tabelas Silver: total de registros e última carga
SELECT 
  'LEITURA_PLUVIOMETROS' as tabela,
  COUNT(*) as total_registros,
  MAX(data_carga_silver) as ultima_carga_silver
FROM silver.leitura_pluviometros

UNION ALL

SELECT 
  'MEDICAO_BACIAS' as tabela,
  COUNT(*) as total_registros,
  MAX(data_carga_silver) as ultima_carga_silver
FROM silver.medicao_bacias

UNION ALL

SELECT 
  'MEDICAO_LAJE' as tabela,
  COUNT(*) as total_registros,
  MAX(data_carga_silver) as ultima_carga_silver
FROM silver.medicao_laje

ORDER BY tabela;

In [0]:
%sql
-- Amostra de leitura_pluviometros (primeiras 10 linhas)
-- Verificar: 
--      data_registro como DATE, 
--      milimetros como DOUBLE, 
--      nomes em lowercase

SELECT unidade, estacao, data_registro, milimetros
FROM mvpDataEngineer.silver.leitura_pluviometros LIMIT 10;

In [0]:
%sql
-- Amostra de medicao_bacias (primeiras 10 linhas)
-- Verificar: 
--      valores "-" agora são NULL, 
--      salmoura_densidade como DOUBLE

SELECT bacia, data_registro, agua_doce_colocado, agua_doce_retirado, 
       salmoura_colocado, salmoura_retirado, salmoura_densidade
FROM mvpDataEngineer.silver.medicao_bacias LIMIT 10;

In [0]:
%sql
-- Amostra de medicao_laje (primeiras 10 linhas)
-- Verificar: data_medicao como DATE, nomes das réguas em lowercase
SELECT data_medicao, cristalizador, area_hectares, 
       regua_1, regua_2, regua_3, regua_4
FROM mvpDataEngineer.silver.medicao_laje LIMIT 10;

In [0]:
%sql
-- Identificar registros duplicados na tabela silver.leitura_pluviometros
-- Agrupa por todas as colunas de negócio e conta quantas vezes cada combinação aparece
SELECT 
  unidade,
  estacao,
  data_registro,
  milimetros,
  COUNT(*) AS qtd_ocorrencias
FROM mvpDataEngineer.silver.leitura_pluviometros
GROUP BY 
  unidade, estacao, data_registro, milimetros
HAVING COUNT(*) > 1
ORDER BY qtd_ocorrencias DESC;

In [0]:
%sql
-- Identificar registros duplicados na tabela silver.medicao_bacias
-- Agrupa por todas as colunas de negócio e conta quantas vezes cada combinação aparece
SELECT 
  unidade,
  bacia,
  data_registro,
  agua_doce_colocado,
  agua_doce_retirado,
  salmoura_colocado,
  salmoura_retirado,
  salmoura_densidade,
  COUNT(*) AS qtd_ocorrencias
FROM mvpDataEngineer.silver.medicao_bacias
GROUP BY 
  unidade, bacia, data_registro, 
  agua_doce_colocado, agua_doce_retirado, salmoura_colocado, salmoura_retirado, salmoura_densidade
HAVING COUNT(*) > 1
ORDER BY qtd_ocorrencias DESC;

In [0]:
%sql
-- Identificar registros duplicados na tabela silver.medicao_laje
-- Agrupa por todas as colunas de negócio e conta quantas vezes cada combinação aparece
SELECT 
  unidade,
  data_medicao,
  cristalizador,
  area_hectares,
  regua_1,  regua_2,  regua_3,  regua_4,  regua_5,  regua_6,  regua_7,  regua_8,
  COUNT(*) AS qtd_ocorrencias
FROM mvpDataEngineer.silver.medicao_laje
GROUP BY 
  unidade, data_medicao, cristalizador, area_hectares,
  regua_1, regua_2, regua_3, regua_4, regua_5, regua_6, regua_7, regua_8
HAVING COUNT(*) > 1
ORDER BY qtd_ocorrencias DESC;

In [0]:
# =========================================
# DEDUPLICAÇÃO: medicao_laje
# =========================================
# Regra de negócio: só pode existir uma leitura por dia (data_medicao) por cristalizador.
# Foram identificados registros duplicados (mais de um protocolo com a mesma leitura).
# Solução: manter apenas um registro por combinação (data_medicao, cristalizador),
# preferindo o protocolo de menor número (mais antigo).

from pyspark.sql.functions import row_number, count as count_fn
from pyspark.sql.window import Window

# Ler a tabela Silver já gravada
df_silver = spark.table("mvpDataEngineer.silver.medicao_laje")

total_antes = df_silver.count()
print(f"Registros antes da deduplicação: {total_antes}")

# Janela particionada por data_medicao + cristalizador, ordenada por protocolo
# para manter sempre o mesmo registro (protocolo de menor número)
w = Window.partitionBy("data_medicao", "cristalizador").orderBy("protocolo")

df_dedup = (df_silver.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn"))

total_depois = df_dedup.count()
duplicados_removidos = total_antes - total_depois
print(f"Registros depois da deduplicação: {total_depois}")
print(f"Duplicados removidos: {duplicados_removidos}")

# Sobrescrever a tabela Silver sem os duplicados
df_dedup.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("mvpDataEngineer.silver.medicao_laje")

print("Tabela mvpDataEngineer.silver.medicao_laje atualizada sem duplicatas.")

# Verificação: não deve haver mais duplicatas
w_check = Window.partitionBy("data_medicao", "cristalizador")
df_verificacao = (
    spark.table("mvpDataEngineer.silver.medicao_laje")
    .withColumn("qtd", count_fn(lit(1)).over(w_check))
    .filter(col("qtd") > 1)
)
restantes = df_verificacao.count()
print(f"Duplicatas restantes após deduplicação: {restantes}")

## 📝 Etapa 3: Documentação das Tabelas e Colunas

Comentários no Unity Catalog tornam os dados autodescritivos, visíveis no Catalog Explorer, autocomplete do SQL Editor e APIs.

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS DA TABELA: leitura_pluviometros
-- =========================================

COMMENT ON TABLE mvpDataEngineer.silver.leitura_pluviometros IS 'Leituras pluviométricas diárias com tipos padronizados: datas como DATE, milímetros como DOUBLE, nomes de colunas em snake_case';

COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.unidade IS 'Unidade federativa de origem (ex: RN)';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.estacao IS 'Nome da estação pluviométrica';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.data_registro IS 'Data do registro da leitura (DATE)';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.milimetros IS 'Quantidade de chuva medida em milímetros (DOUBLE)';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.data_ingestao IS 'Timestamp da ingestão na camada Staging';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.origem IS 'Origem dos dados (google_drive)';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.arquivo_origem IS 'Nome do arquivo Excel de origem';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.data_carga_bronze IS 'Timestamp da carga na camada Bronze';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.origem_tabela IS 'Tabela de staging de origem';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.data_carga_silver IS 'Timestamp da carga na camada Silver';
COMMENT ON COLUMN mvpDataEngineer.silver.leitura_pluviometros.origem_tabela_bronze IS 'Tabela de bronze de origem';

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS DA TABELA: medicao_bacias
-- =========================================

COMMENT ON TABLE mvpDataEngineer.silver.medicao_bacias IS 'Medições das bacias evaporadoras com tipos padronizados: volumes como INT, densidade como DOUBLE, valores ausentes como NULL';

COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.unidade IS 'Unidade federativa de origem (ex: RN)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.bacia IS 'Identificador da bacia evaporadora';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.data_registro IS 'Data do registro da medição (DATE)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.agua_doce_colocado IS 'Volume de água doce colocada na bacia (INT, NULL se ausente)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.agua_doce_retirado IS 'Volume de água doce retirada da bacia (INT, NULL se ausente)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.salmoura_colocado IS 'Volume de salmoura colocada na bacia (INT, NULL se ausente)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.salmoura_retirado IS 'Volume de salmoura retirada da bacia (INT, NULL se ausente)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.salmoura_densidade IS 'Densidade da salmoura em g/cm³ (DOUBLE)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.data_ingestao IS 'Timestamp da ingestão na camada Staging';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.origem IS 'Origem dos dados (google_drive)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.arquivo_origem IS 'Nome do arquivo Excel de origem';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.data_carga_bronze IS 'Timestamp da carga na camada Bronze';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.origem_tabela IS 'Tabela de staging de origem';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.data_carga_silver IS 'Timestamp da carga na camada Silver';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_bacias.origem_tabela_bronze IS 'Tabela de bronze de origem';

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS DA TABELA: medicao_laje
-- =========================================

COMMENT ON TABLE mvpDataEngineer.silver.medicao_laje IS 'Medições da laje dos cristalizadores com tipos padronizados: data de medição como DATE, réguas como INT, nomes em snake_case';

COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.unidade IS 'Unidade federativa de origem (ex: RN)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.data_medicao IS 'Data da medição na laje (DATE)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.cristalizador IS 'Número de identificação do cristalizador';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.area_hectares IS 'Área do cristalizador em hectares (DOUBLE)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_1 IS 'Leitura da régua 1 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_2 IS 'Leitura da régua 2 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_3 IS 'Leitura da régua 3 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_4 IS 'Leitura da régua 4 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_5 IS 'Leitura da régua 5 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_6 IS 'Leitura da régua 6 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_7 IS 'Leitura da régua 7 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.regua_8 IS 'Leitura da régua 8 de medição de nível (INT)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.data_ingestao IS 'Timestamp da ingestão na camada Staging';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.origem IS 'Origem dos dados (google_drive)';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.arquivo_origem IS 'Nome do arquivo Excel de origem';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.data_carga_bronze IS 'Timestamp da carga na camada Bronze';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.origem_tabela IS 'Tabela de staging de origem';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.data_carga_silver IS 'Timestamp da carga na camada Silver';
COMMENT ON COLUMN mvpDataEngineer.silver.medicao_laje.origem_tabela_bronze IS 'Tabela de bronze de origem';